In [ ]:
import datetime as dt

from cartopy.crs import NorthPolarStereo
from cartopy.feature import LAND, COASTLINE
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

from pysida.lib import get_rgps_pairs, get_deformation_from_pair

In [ ]:
def decorate_map(ax, map_extent, crs, title):
    ax.add_feature(LAND)
    ax.add_feature(COASTLINE)
    ax.set_extent(map_extent, crs=crs)
    ax.set_title(title)

In [ ]:
df = pd.read_pickle('../rgps_csv/w07_may_LP.df')

In [ ]:
date0 = dt.datetime(2007,1,1)
date1 = dt.datetime(2007,1,6)
min_time_diff = 0.5
max_time_diff = 3.1
min_size = 800
r_min = 0.1
a_max = 400e6
cores = 5

pairs = get_rgps_pairs(
    df, date0, date1,
    min_time_diff=min_time_diff,
    max_time_diff=max_time_diff,
    min_size=min_size,
    r_min=r_min,
    a_max=a_max,
    cores=cores
)
len(pairs)

In [ ]:
srs_dst = NorthPolarStereo(central_longitude=-45, true_scale_latitude=60)

x_lft=-2500000.0
x_rht=300000
y_top=2500000
y_bot=-1000000.0
map_extent0 = [-2300000, -600000, -500000, 1000000]
map_extent1 = [-2300000, -600000, -500000, 1600000]
figsize = (14,6)

fig, ax = plt.subplots(1,3, figsize=figsize, subplot_kw={'projection': srs_dst})

# plot trajectories
df_g = df.g.to_numpy()
dfg_unique, dfg_sizes = np.unique(df_g, return_counts=True)
dfg_unique = dfg_unique[dfg_sizes > 80]
for g in dfg_unique[::5]:
    time_diff = (df.d - pd.Timestamp('2006-12-01'))
    time_diff_seconds = time_diff.dt.total_seconds() / (24 * 3600)
    gpi0 = df.g == g

    ax[0].plot(df[gpi0].x, df[gpi0].y, 'k-', alpha=0.5, zorder=1)
    scat0 = ax[0].scatter(df[gpi0].x, df[gpi0].y, 10, time_diff_seconds[gpi0], clim=[0, 180], zorder=2)
decorate_map(ax[0], map_extent0, srs_dst, 'A. Few longest RGPS trajectories in 2006/2007')
plt.colorbar(scat0, ax=ax[0], label='Days since 2006-12-01', orientation='horizontal', shrink=0.5)


for p in pairs:
    ax[1].plot(p.x0, p.y0, '.', label=f'{str(p.d0)[:19]} - {str(p.d1)[:19]}', ms=7)
decorate_map(ax[1], map_extent1, srs_dst, 'B. RGPS buoy positions, 1 - 5 Jan 2007')
ax[1].legend()

for p in pairs:
    e = get_deformation_from_pair(p)
    trp = ax[2].tripcolor(p.x0, p.y0, p.t, e.e2*24*60*60*100, mask=~p.g, clim=[0, 10], cmap='plasma_r')
decorate_map(ax[2], map_extent0, srs_dst, 'C. RGPS Shear, 1 - 5 Jan 2007')
plt.colorbar(trp, ax=ax[2], label='Shear, 100%/day', orientation='horizontal', shrink=0.5)

plt.tight_layout()
plt.savefig('../tuning_paper_figures/fig_01_rgps.png', dpi=150)
plt.show()